# WidowX MuJoCo Frozen CLIP Patch Pointer V2
This private run trains only the frozen-CLIP spatial pointer head. Its output must be returned to the local MuJoCo evaluator; Kaggle validation is not a closed-loop result.


In [ ]:
from pathlib import Path
import json
import zipfile

input_root = Path('/kaggle/input')
manifest_paths = list(input_root.glob('*/manifest.json'))
archive_paths = list(input_root.glob('*/kaggle_patch_pointer_v2.zip'))
if len(manifest_paths) == 1:
    pack_root = manifest_paths[0].parent
elif len(archive_paths) == 1:
    pack_root = Path('/kaggle/working/kaggle_patch_pointer_v2')
    with zipfile.ZipFile(archive_paths[0]) as archive:
        archive.extractall(pack_root)
else:
    raise RuntimeError(f'Expected one attached manifest or pack archive, found manifests={manifest_paths}, archives={archive_paths}')
if (pack_root / 'kaggle_patch_pointer_v2.zip').exists():
    archive_path = pack_root / 'kaggle_patch_pointer_v2.zip'
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(pack_root)
manifest = json.loads((pack_root / 'manifest.json').read_text())
assert manifest['samples'] == 393
print(manifest['version'], manifest['dataset_content_sha256'])

!pip install -q transformers==4.57.0
!python {pack_root}/scripts/kaggle_train_patch_pointer.py --dataset-root {pack_root} --output /kaggle/working/clip_patch_pointer_kaggle_v2.pt --metrics /kaggle/working/clip_patch_pointer_kaggle_v2.json --epochs 300 --device cuda
